# AntiSlop API Server Example

This notebook demonstrates using AntiSlop as an OpenAI-compatible API server.

## 1. Start the Backend (vLLM)

Run this in a terminal:
```bash
vllm serve unsloth/gemma-3-4b-it --port 8000 --api-key xxx
```

## 2. Start the AntiSlop Server

Run this in another terminal:
```bash
python main.py --openai-api --openai-api-port 8080 \
    --api-base-url "http://localhost:8000/v1" \
    --api-key "xxx" \
    --model-name "unsloth/gemma-3-4b-it" \
    --chat-template-model-id "unsloth/gemma-3-4b-it" \
    --slop-phrases-file "banlists/slop_phrases.json" \
    --top-n-slop-phrases 500 \
    --regex-blocklist-file "banlists/regex_not_x_but_y.json" \
    --temperature 1.0 \
    --min-p 0.03 \
    --ban-strength 1.0
```

## 3. Query the Server

In [ ]:
from openai import OpenAI

# Point to the AntiSlop server
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="unused"  # not validated by AntiSlop server
)

In [ ]:
response = client.chat.completions.create(
    model="unsloth/gemma-3-4b-it",
    messages=[
        {"role": "user", "content": "Write a short story about a princess."}
    ],
    max_tokens=500,
    temperature=0.9,
)

print(response.choices[0].message.content)

## 4. Using requests directly

In [ ]:
import requests

resp = requests.post(
    "http://localhost:8080/v1/chat/completions",
    json={
        "model": "unsloth/gemma-3-4b-it",
        "messages": [{"role": "user", "content": "Explain quantum computing briefly."}],
        "max_tokens": 300,
        "temperature": 0.7,
        "min_p": 0.05,
        "top_p": 0.95,
        "top_k": 50,
        "ban_strength": 1.0
    }
)

print(resp.json()["choices"][0]["message"]["content"])

## Supported Request Parameters

| Parameter | Description |
|-----------|-------------|
| `messages` | Chat messages (required) |
| `max_tokens` | Max tokens to generate |
| `temperature` | Sampling temperature |
| `min_p` | Min-p sampling |
| `top_p` | Nucleus sampling |
| `top_k` | Top-k filtering |
| `ban_strength` | 0-1, soft to hard ban (default 1.0) |

**Server-side only** (set via CLI when starting server):
- `stop` sequences, ban lists, backtracking config